
Tren Meckel, Tanvi Shegaonkar <br>
Dr. Truex<br>
CS 323: Data Privacy<br>
12 September 2025<br>


# Project 1 : Anonymity 

## Import statements 

In [2]:
import os
import math
import pandas as pd
import numpy as np
from scipy.stats import entropy
import itertools

### Small setup for the data

In [4]:
#Load the dataset   
file = pd.read_csv("survey.csv")
df = pd.DataFrame(file)


if "Timestamp" in df.columns:           # Don't need
    df = df.drop(columns=["Timestamp"])
if "comments" in df.columns:            # Sensitive attribute therefore --> suppressed 
    df = df.drop(columns=["comments"])
if "Unnamed: 3" in df.columns: 
    df = df.drop(columns=["Unnamed: 3"]) # bye bye gender clean


### Before beginning
 As this dataset contained actual survey responses, the data required some cleaning before it was able to be anonymized. Several fields such as Age, Country, and Gender were designed as open-ended responses, resulting in rather messy data including nonsensical answers (Age = 9999999) and typographical errors (femail rather than female). We recognized gender as an open-ended field so we considered all responses as legitimate responses.

In [6]:
'''
Clean-Up for Gender and Age:
1. Normalize text entries in Gender for consistency.
2. Categorize Gender into broader groups (Male, Female, Other) to reduce uniqueness.
3. Convert Age to numeric, handle invalid values, and prepare for generalization.
'''

# --- Step 1: Normalize Gender ---     #NOT IN THAT WAY! JUST FOR THE SAKE OF THE DATA :(
df["Gender"] = df["Gender"].astype(str).str.lower().str.strip()


# --- Step 2: Define lists of gender terms --- (mainly male has some interesting answers)
male_terms = ["male","something kinda male?", "make","msle", "mail", "male-ish",
              "ostensibly male, unsure what that really means", "m", "cis man", 
              "cis male", "male (cis)", "maile", "guy (-ish) ^_^", "Male-ish", 
              "man", "malr", "mal"]

female_terms = ["female", "femle", "f", "woman", "cis-female", "cis female", 
                "female (cis)", "trans woman", "cis-female/femme", "trans-female",
                "femake", "cis woman","female (trans)" ,"femail"]

#===========Helper function: clean_gender===========
def clean_gender(val):
    if val in male_terms:
        return "Male"
    elif val in female_terms:
        return "Female"
    else:
        return "Other"

# Apply new categories of generalized gender to replace previous responses in the Gender column
df["Gender"] = df["Gender"].apply(clean_gender)

# --- Step 4: Convert Age to numeric ---
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

# Replace invalid ages with "Unknown"
df.loc[(df["Age"] < 12) | (df["Age"] > 100), "Age"] = None  # temporary placeholder

# Convert whole column to string so we can mix numbers + "Unknown"
df["Age"] = df["Age"].astype("Int64").astype(str)
df.loc[df["Age"].isin(["<NA>", "nan"]), "Age"] = "Unknown"

# --- Step 5: Save the cleaned dataset ---
df.to_csv("survey_data_clean.csv", index=False)

## Still cleaning...
 With other fields such as Age and Country, we decided that we needed a legitimate response for both fields for a record to be considered to be complete. If these two fields were left incomplete or had a nonsensical response, we discarded that row. After removing incomplete records, the dataset contained 1251 records. This tidied version of the raw data is attached in survey.csv.

In [8]:
'''
Clean up for Country and State columns:
1. Normalize text formatting for consistency.
2. Ensure only U.S. rows have valid states; others have NaN.
3. Validate data and mark any inconsistencies.
4. Save cleaned DataFrame for further processing.
'''

# --- Step 1: Normalize Country and State strings ---
df["Country"] = df["Country"].astype(str).str.strip().str.title()
df["state"] = df["state"].astype(str).str.strip().str.upper()

# --- Step 2: Define valid reference lists ---
valid_us_states = [
    "AL","AK","AZ","AR","CA","CO","CT","DE","DC","FL","GA","HI","ID","IL","IN","IA","KS",
    "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC",
    "PR","GU","AS","VI","MP"
]

us = {"United States"}

# known valid countries
valid_countries = set(us) | {
    "Canada","Mexico","United Kingdom","Germany","France","Spain","Italy","Australia",
    "India","China","Japan","Brazil","Bahamas","Bulgaria","Portugal","Netherlands",
    "Switzerland","Poland","Russia","Slovenia","Costa Rica","Austria","Ireland",
    "South Africa","Sweden","Colombia","Latvia","Romania","Belgium","New Zealand",
    "Zimbabwe","Finland","Uruguay","Israel","Bosnia And Herzegovina","Hungary",
    "Singapore","Nigeria","Croatia","Norway","Thailand","Denmark","Greece","Moldova",
    "Georgia","Czech Republic","Philippines"
}

# placeholders for empty states
empty_state_values = {"", "NAN", "NA", "N/A", "NONE", "NULL", "<NA>"}

# --- Step 3: Fill missing state values based on Country ---
# take care U.S. rows with empty states --> set to "Unknown"
mask_us_na = df["Country"].isin(us) & df["state"].isin(empty_state_values)
df.loc[mask_us_na, "state"] = "Unknown"

# non-U.S. rows with empty --> set to NaN
mask_non_us_na = ~df["Country"].isin(us) & df["state"].isin(empty_state_values)
df.loc[mask_non_us_na, "state"] = np.nan

#===========Define function to validate Country-State consistency===========
'''
check_country_state(row):
    Validates each row based on:
        - U.S. rows must have valid states or "Unknown"
        - Non-U.S. rows must have NaN for state
        - Unrecognized countries marked as invalid
Outputs:
    "Valid" or "Invalid" for each row
'''
def check_country_state(row):
    country = row["Country"]
    state = row["state"]

    # U.S. --> state must be valid or "Unknown"
    if country in us:
        if state not in valid_us_states and state != "Unknown":
            return "Invalid"

    # Non-U.S. --> state must be NaN
    elif country in valid_countries:
        if not pd.isna(state):
            return "Invalid"

    # Country not recognized --> invalid
    else:
        return "Invalid"

    return "Valid"

# --- Step 4: Apply validation ---
df["Country_State_Check"] = df.apply(check_country_state, axis=1)

# --- Step 5: Correct invalid entries ---
for idx, row in df.iterrows():
    if row["Country_State_Check"] == "Invalid":
        if row["Country"] not in valid_countries and row["Country"] not in us:
            df.at[idx, "Country"] = "Unknown"
            df.at[idx, "state"] = np.nan
        else:
            # valid country but nonsense state --> just clear state
            df.at[idx, "state"] = np.nan


# --- Step 6: Save cleaned DataFrame ---
df.to_csv("survey_data_clean.csv", index=False)

In [9]:
# --- Drop temporary validation column ---
if "Country_State_Check" in df.columns:           # Don't need
    df = df.drop(columns=["Country_State_Check"])

## 'Unknown' Values
This is where all rows marked with 'Unknown' are dropped in the dataset along with we decided in order to anonymie the data, so people living in the United States, in order to not be idenified with states and the column asking how many people they work with in their company, we dropped the state column as well to not deal with people being uniquely idenfied and having the column become a QID as well. We also found that keeping the column of how many people they worked with in their company to be more important as that showed in some people's cases to be a reason to affect their mental health in Tech.

In [16]:
'''
Data cleanup before generalization/anonymization:
1. Drop unnecessary or sensitive columns.
2. Remove rows with unknown critical QID values.
3. Reset DataFrame index after row removals.
4. Ensure Age is numeric for subsequent generalization.
'''

# --- Step 1: Drop the 'state' column ---
if "state" in df.columns:
    df = df.drop(columns=["state"])

# --- Step 2: Drop rows with unknown Age or Country ---
df = df[~(
    (df["Age"] == "Unknown") |                        
    (df["Country"] == "Unknown") #| 
)]

# --- Step 3: Reset DataFrame index ---
df = df.reset_index(drop=True)

# --- Step 4: Ensure Age column is numeric ---
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

# --- Step 5: Save cleaned dataset ---
df.to_csv("survey_data_clean.csv", index=False)

## k = 2 Anonymity


Generalization for achieving k-anonymity:<br>
1. Generalize Country into Regions <br>
2. Generalize Age into ranges to reduce granularity.<br>
3. Combine Gender categories ("Female" + "Other") to reduce uniqueness.<br>
Goal: ensure 100% anonymity (no unique QID combinations).<br>

We utilize the inbuilt pandas duplicated function to check how many rows have a duplicate. These rows are considered to be k=2 anonymous.

In [20]:
#===========Helper function: Generalize Country===========
'''
generalize_country
Groups individual countries into broader regions to reduce uniqueness
and aid in anonymization.

Inputs:
    country: string representing a country name from the dataset

Outputs:
    generalized_country: string representing either:
        - Original country if kept as-is (US, UK, Canada)
        - A region label ("Africa/Asia/Oceania/Americas", "Europe")
        - "Unknown" if country does not match any known categories
'''
def generalize_country(country):
    if country == "United States":
        return "United States"
    elif country == "United Kingdom":
        return "United Kingdom"
    elif country == "Canada":
        return "Canada"
    elif country in ["South Africa","Nigeria","Zimbabwe","India","China","Japan","Singapore"
                     ,"Thailand","Philippines","Israel", "Australia","New Zealand", 
                     "Mexico","Costa Rica","Bahamas","Brazil","Uruguay","Colombia"]:
        return "Africa/Asia/Americas"
    elif country in ["Germany","Netherlands","Ireland","France","Switzerland","Belgium","Austria",
                    "Sweden","Finland","Norway","Latvia", "Italy","Spain","Portugal","Greece",
                     "Croatia","Slovenia","Bulgaria", "Czech Republic","Denmark", "Poland", "Russia", 
                     "Georgia", "Bosnia And Herzegovina","Romania","Moldova","Hungary"]:
        return "Europe"
    else:
        return "Unknown"

In [22]:
df_k2 = df.copy(deep=True)

# --- Step 1: Generalize Country into Regions ---
df_k2["Country"] = df_k2["Country"].apply(generalize_country)

# --- Step 2: Generalize Age into Age Ranges ---
bins = pd.IntervalIndex.from_tuples([(17,26),(26,30),(30,50),(50,100)])
label = ["17-26", "27-30", "31-50", "51+"]
agerange = pd.cut(df.Age, bins, right=False, labels=label)      # Bin ages
df_k2.insert(0, "Age Range", agerange)                     # Insert new column at the front

# --- Step 3: Combine Female & Other Gender Categories (overwrite Gender) ---
mask = df_k2["Gender"].isin(["Female","Other"])
df_k2.loc[mask, "Gender"] = "Female/Other"
df_k2.loc[~mask, "Gender"] = "Male"

# --- Step 4: Identify duplicates to ensure k-anonymity ---
duplicates = df_k2.duplicated(
    subset=["Age Range","Gender","Country"], 
    keep=False
)

# Extract fully anonymized (duplicate) rows
df_anon = df_k2[duplicates].copy()

# Drop the original Age column (optional, if you only want Age Range)
if "Age" in df_anon.columns:
    df_anon = df_anon.drop(columns=["Age"])

# Sort for readability
df_anon = df_anon.sort_values(by=["Age Range","Gender","Country"])

# Save anonymized dataset
df_anon.to_csv("Final_Anon_Survey_k2.csv", index=False)

# Extract rows that are still unique (not 2-anonymized)
df_not_anon = df_k2[~duplicates].copy()
df_not_anon = df_not_anon.sort_values(by=["Age Range","Gender","Country"])
df_not_anon.to_csv("Not_Anon_Survey_k2.csv", index=False)

## Anonymization Validation Function


In [25]:
'''
get_k_anonymized (df, k)

Inputs:  Dataframe df, int k
Outputs: Dataframe dk_k  with k-anonymized Values, Dataframe df_not_k with not k-anonymized values
'''
# --- Load anonymized dataset you already prepared ---
df0 = pd.read_csv("Final_Anon_Survey_k2.csv")

# --- Function to get k-anonymized dataset ---
def get_k_anonymized(df, k):
    """
    Keeps only rows where the combination of QIDs has at least k duplicates.
    QIDs: Age Range, Gender Combined, Country
    """
    # Compute group sizes for each combination of QIDs
    group_sizes = df.groupby(['Age Range','Gender','Country'], observed=False).size()
    # Keep only rows that belong to groups of size >= k
    mask = df.apply(
        lambda row: group_sizes.loc[(row['Age Range'], row['Gender'], row['Country'])] >= k,
        axis=1
    )
    
    df_k = df[mask]
    df_not_k = df[~mask]
    df_not_k.reset_index(drop = True, inplace = True)
    df_k.reset_index(drop=True, inplace=True)
    return df_k, df_not_k


## K = 3 Anonymity


 Upon testing from k=2 dataset, the following equivalence sets only had two members and were not k=3 anonymous: <br>
 Female/Other, 30-50, Africa/Asia/Americas - Isolated by Country and Age  (34, 38) <br>  Female/Other, 50-100, United States - Isolated by Age (56, 72)<br> 

Solution: Rebin Ages into (17,26],  (26,44],  (44, 100]

In [29]:
file = pd.read_csv("survey_data_clean.csv")
df_k3 = pd.DataFrame(file)

In [31]:
df_k3 = df.copy(deep=True)

# --- Step 1: Generalize Country into Regions ---
df_k3["Country"] = df_k3["Country"].apply(generalize_country)

# --- Step 2: Generalize Age into Age Ranges ---

# Define bins and labels # 
label = ["17-26", "27-44", "45-100"]  # because it will be if the categories don't match
rebin = pd.IntervalIndex.from_tuples([(17,26),(26,44),(44,100)])  # Interval ranges (18,26),(27,30),(31,50),(51,100)
ranges = pd.cut(df['Age'], rebin, right=False, labels = label)      # Bin ages
df_k3.insert(0, "Age Range", ranges)                     # Insert new column at the front

# --- Step 3: Combine Female & Other Gender Categories (overwrite Gender) ---
mask = df_k3["Gender"].isin(["Female","Other"])
df_k3.loc[mask, "Gender"] = "Female/Other"
df_k3.loc[~mask, "Gender"] = "Male"

# --- Step 4: Verify all rows are k=3 anonymized rows ---
df_k3, df_not_k3 = get_k_anonymized(df_k3, 3)

# Drop raw Age column 
if "Age" in df_k3.columns:
    df_k3 = df_k3.drop(columns=["Age"])

# Sort for readability using only generalized QIDs
df_k3 = df_k3.sort_values(by=["Age Range","Gender","Country"])

# Save final datasets
df_k3.to_csv("Final_Anon_Survey_k3.csv", index=False)
df_not_k3.to_csv("Not_Anon_Survey_k3.csv", index=False)

## K = 4 Anonymity


 Upon testing from k=3 dataset, the following equivalence set only had three members and was not k=4 anonymous: <br>
 Male, 44-100, Europe

Solution: Additional male records within this age range exist in the United Kingdom. To ensure k=2 anonymity, we can include United Kingdom as part of the region Europe.

In [35]:
file = pd.read_csv("survey_data_clean.csv")
df_k4 = pd.DataFrame(file)

In [37]:
df_k4 = df.copy(deep=True)

# --- Step 1: Generalize Country into Regions ---
df_k4["Country"] = df_k4["Country"].apply(generalize_country)

# --- Step 2: Include United Kingdom into Europe ---
df_k4["Country"] = df_k4["Country"].replace("United Kingdom", "Europe")

# --- Step 3: Generalize Age into Age Ranges ---

# Define bins and labels # 
label = ["17-26", "27-44", "45-100"]  # because it will be if the categories don't match
rebin = pd.IntervalIndex.from_tuples([(17,26),(26,44),(44,100)])  # Interval ranges (18,26),(27,30),(31,50),(51,100)
ranges = pd.cut(df['Age'], rebin, right=False, labels = label)      # Bin ages
df_k4.insert(0, "Age Range", ranges)                     # Insert new column at the front

# --- Step 4: Combine Female & Other Gender Categories (overwrite Gender) ---
mask = df_k4["Gender"].isin(["Female","Other"])
df_k4.loc[mask, "Gender"] = "Female/Other"
df_k4.loc[~mask, "Gender"] = "Male"

# --- Step 5: Verify all rows are k=4 anonymized rows ---
df_k4, df_not_k4 = get_k_anonymized(df_k4, 4)

# Drop raw Age column
if "Age" in df_k4.columns:
    df_k4 = df_k4.drop(columns=["Age"])

# Sort for readability using only generalized QIDs
df_k4 = df_k4.sort_values(by=["Age Range","Gender","Country"])

# Save final datasets
df_k4.to_csv("Final_Anon_Survey_k4.csv", index=False)
df_not_k4.to_csv("Not_Anon_Survey_k4.csv", index=False)

## Utility Function
We chose to measure utility for our anonymized datasets using KL- Divergence. 
This was done by utilizing n

In [40]:
#===========Helper: KL divergence with smoothing===========
'''
kl_divergence
Computes the KL divergence between two probability distributions.
Adds a tiny epsilon to avoid log(0) errors.

Inputs:
    p: array-like of probabilities for the original distribution
    q: array-like of probabilities for the anonymized distribution
    eps: small value added to avoid zero probabilities (default 1e-12)

Outputs:
    kl: KL divergence (nat/log base e) as a float
'''
def kl_divergence(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps # scipy.stats.entropy(p, q) computes KL divergence KL(p||q)
    q = np.asarray(q, dtype=float) + eps # Formula: sum_i p[i] * log(p[i] / q[i])
    return entropy(p, q)  # KL(p || q)

#===========Helper: align_distributions===========
'''
align_distributions
Aligns two distributions so that each category has a matching index.
Fills missing categories with probability 0.

Inputs:
    orig_series: Pandas Series of the original dataset's QID
    anon_series: Pandas Series of the anonymized dataset's QID

Outputs:
    orig_aligned: Pandas Series of probabilities for original dataset, aligned
    anon_aligned: Pandas Series of probabilities for anonymized dataset, aligned
'''
def align_distributions(orig_series, anon_series):

    # counts the frequencies of each category --> normalize to relative frequencies 
    orig_counts = orig_series.value_counts(normalize=True).sort_index()  
    anon_counts = anon_series.value_counts(normalize=True).sort_index()

    # make full set of all categories present in both datasets
    all_cats = sorted(set(orig_counts.index) | set(anon_counts.index))

    # reindex both distributions to the full set, missing become 0s
    orig_aligned = orig_counts.reindex(all_cats, fill_value=0)
    anon_aligned = anon_counts.reindex(all_cats, fill_value=0)

    return orig_aligned, anon_aligned

#===========Main utility function (KL only)===========
def measure_qid_utility_kl_only(orig_df, anon_df, qids, save_csv="kl_results.csv"):
    """
    measure_qid_utility_kl_only
    Measures utility of anonymized dataset by computing KL divergence
    between original and anonymized distributions for given QIDs.
    Saves a single CSV with results.

    Inputs:
        orig_df: Pandas DataFrame of original dataset
        anon_df: Pandas DataFrame of anonymized dataset
        qids: list of strings, names of quasi-identifiers (columns) to evaluate
        save_csv: filename for saving KL divergence results

    Outputs:
        all_results: list of tuples (qid, kl_divergence) for each QID
    """
    results_list = [] # empty list to put kl results in

    for qid in qids:
        print(f"Processing QID: {qid}")
        orig_aligned, anon_aligned = align_distributions(orig_df[qid], anon_df[qid]) # align the distributions
        kl = kl_divergence(orig_aligned, anon_aligned) # compute the kl of the aligned distributions
        print(f"  KL divergence: {kl:.6f}")
        results_list.append({"QID": qid, "KL_divergence": kl}) #kl result are now in

    # Save all QID KL divergences in a single CSV
    results_df = pd.DataFrame(results_list)
    results_df.to_csv(save_csv, index=False)
    print(f"\nKL divergence results saved to: {save_csv}")
    
    return results_list


In [50]:
'''
k=2 Utility 
'''
orig_k2 = pd.read_csv("survey_data_clean.csv")
anon_k2 = pd.read_csv("Final_Anon_Survey_k2.csv")

# Re-bin ages to match anonymized dataset
bins = [17, 27, 31, 51, 100]            # need to do this so the bins match the categories so KL isn't too big
label = ["17-26", "27-30", "31-50", "51+"]  # because it will be if the categories don't match
orig_k2["Age Range"] = pd.cut(orig_k2["Age"], bins=bins, right=False, labels=label)


qids = ["Age Range", "Gender", "Country"] #QIDS that were anonymized

kl_results_k2 = measure_qid_utility_kl_only(orig_k2, anon_k2, qids, save_csv="kl_results_k2.csv")


Processing QID: Age Range
  KL divergence: 26.531881
Processing QID: Gender
  KL divergence: 5.422932
Processing QID: Country
  KL divergence: 4.569574

KL divergence results saved to: kl_results_k2.csv


In [43]:
'''
k=3 and k=4 Utility
'''

orig_k34 = pd.read_csv("survey_data_clean.csv")
anon_k3 = pd.read_csv("Final_Anon_Survey_k3.csv")
anon_k4 = pd.read_csv("Final_Anon_Survey_k4.csv")

bins = [17, 27, 45, 100]            # need to do this so the bins match the categories so KL isn't too big
label = ["17-26", "27-44", "45-101"]  # because it will be if the categories don't match
orig_k34["Age Range"] = pd.cut(orig_k34["Age"], bins=bins, right=False, labels=label)

qids = ["Age Range", "Gender", "Country"] #QIDS that were anonymized

kl_k3 = measure_qid_utility_kl_only(orig_k34, anon_k3, qids, save_csv="kl_results_k3.csv")
kl_k4 = measure_qid_utility_kl_only(orig_k34, anon_k4, qids, save_csv="kl_results_k4.csv")

Processing QID: Age Range
  KL divergence: 26.891423
Processing QID: Gender
  KL divergence: 5.422932
Processing QID: Country
  KL divergence: 4.569574

KL divergence results saved to: kl_results_k3.csv
Processing QID: Age Range
  KL divergence: 26.891423
Processing QID: Gender
  KL divergence: 5.422932
Processing QID: Country
  KL divergence: 8.351687

KL divergence results saved to: kl_results_k4.csv


In [46]:
#===========Helper: Align Distributions for Joint QIDs===========
'''
align_distributions_of_all_QIDs
Computes the KL divergence for multiple QIDs jointly.
Aligns the distributions of all combinations of the given QIDs between original and anonymized datasets.
Fills missing joint categories with 0 probability.

Inputs:
    orig_df: Pandas DataFrame of the original dataset
    anon_df: Pandas DataFrame of the anonymized dataset
    qids: list of strings, names of quasi-identifiers to evaluate jointly

Outputs:
    orig_aligned: Pandas Series of normalized joint probabilities for the original dataset
    anon_aligned: Pandas Series of normalized joint probabilities for the anonymized dataset
'''
def align_distributions_of_all_QIDs(orig_df, anon_df, qids):
    
    # Compute normalized frequency distributions for all QIDs jointly
    orig_counts = orig_df.groupby(qids, observed=False).size().div(len(orig_df))
    anon_counts = anon_df.groupby(qids, observed=False).size().div(len(anon_df))

    # Make sure both distributions have the same set of joint categories
    all_combos = sorted(set(orig_counts.index) | set(anon_counts.index))

    orig_aligned = orig_counts.reindex(all_combos, fill_value=0)
    anon_aligned = anon_counts.reindex(all_combos, fill_value=0)

    return orig_aligned, anon_aligned
    
#===========Main Function: Compute Joint KL Divergence Across k===========
'''
compare_joint_kl_across_k
Measures utility of anonymized datasets by computing KL divergence
for joint QIDs (multiple columns considered together) across different k-anonymity datasets.
Saves a single CSV with results for all k values.

Inputs:
    orig_df: Pandas DataFrame of the original dataset
    anon_datasets: dictionary mapping k values to anonymized DataFrames
    qids: list of strings, names of quasi-identifiers to evaluate jointly
    age_binnings: dictionary mapping k values to age binning rules
    save_csv: filename for saving joint KL divergence results

Outputs:
    results_df: Pandas DataFrame with KL divergence for each k
'''
def compare_joint_kl_across_k(orig_df, anon_datasets, qids, age_binnings, save_csv="kl_joint_results.csv"):
    results = []

    for k_val, anon_df in anon_datasets.items():
        print(f"\n--- Processing k={k_val} ---")
        
        orig_copy = orig_df.copy()
        
        bins = age_binnings[k_val]["bins"]
        labels = age_binnings[k_val]["labels"]
        orig_copy["Age Range"] = pd.cut(orig_copy["Age"], bins=bins, right=False, labels=labels)
        orig_aligned, anon_aligned = align_distributions_of_all_QIDs(orig_copy, anon_df, qids)
        kl = kl_divergence(orig_aligned, anon_aligned)

        print(f"Joint KL divergence for QIDs {qids}: {kl:.6f}")
        results.append({"k": k_val, "Joint_KL": kl})

    results_df = pd.DataFrame(results)
    results_df.to_csv(save_csv, index=False)
    print(f"\nJoint KL divergence results saved to: {save_csv}")
    return results_df

#===========Setup: Age Binnings per k===========
age_binnings = {
    2: {
        "bins": [17, 27, 31, 51, 100],
        "labels": ["17-26", "26-30", "30-50", "50+"]
    },
    3: {
        "bins": [17, 27, 45, 100],
        "labels": ["17-26", "27-44", "45-101"]
    },
    4: {
        "bins": [17, 27, 45, 100],
        "labels": ["17-26", "27-44", "45-101"]
    }
}

#===========Load Anonymized Datasets per k===========
anon_datasets = {
    2: pd.read_csv("Final_Anon_Survey_k2.csv"),
    3: pd.read_csv("Final_Anon_Survey_k3.csv"),
    4: pd.read_csv("Final_Anon_Survey_k4.csv")
}

#===========Load Original Dataset===========
orig_df = pd.read_csv("survey_data_clean.csv")
qids = ["Age Range", "Gender", "Country"]

# Run it!!!!
joint_results = compare_joint_kl_across_k(orig_df, anon_datasets, qids, age_binnings)


--- Processing k=2 ---
Joint KL divergence for QIDs ['Age Range', 'Gender', 'Country']: 24.417195

--- Processing k=3 ---
Joint KL divergence for QIDs ['Age Range', 'Gender', 'Country']: 24.750623

--- Processing k=4 ---
Joint KL divergence for QIDs ['Age Range', 'Gender', 'Country']: 24.750623

Joint KL divergence results saved to: kl_joint_results.csv
